In [13]:
import cx_Oracle
import itertools
import joblib
import shap
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
# import missingno as msno

In [14]:
%matplotlib inline
sns.set_style('whitegrid')
from matplotlib import font_manager
# font_path='/usr/share/fonts/cjkuni-uming/uming.ttc'
font_path = '/usr/share/fonts/truetype/arphic/uming.ttc'
matplotlib.rcParams['font.family']=font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus']=False

In [15]:
import warnings
warnings.filterwarnings('ignore')

In [16]:
%matplotlib inline

### 现金流量表，年报对齐到每天

In [60]:
# 获取现金流量表中的年报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where  REPORT_PERIOD like '%1231' and STATEMENT_TYPE in (408001000,408005000,408027000,408028000,408036000,408045000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [61]:
df1.shape

(62517, 122)

In [62]:
df1.head()

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
0,{F417E2D4-B2DA-43A7-8CBD-64BB0759B996},600015.SH,08M399B152,20070314,20060217,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2018-07-31 10:38:38,0
1,{E3791D76-86FA-4B6B-ABC1-AD174EC6A2AE},600016.SH,1600016,20070319,20060228,20051231,408001000,CNY,NaN,NaN,...,NaN,1.125393e+10,0,NaN,None,NaN,NaN,合并报表,2019-03-19 13:30:52,0
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
3,{96D6CF45-08F7-44BF-B19A-63E987CADBDC},000001.SZ,1000001,20070322,20060401,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2023-08-01 15:59:44,0
4,{3A52DC8F-D3E7-C747-E040-007F01001792},600036.SH,1600036,20060412,20060412,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-19 09:48:07,0


In [69]:
start_date=df1['REPORT_PERIOD'].min()

In [73]:
end_date='20250626'

In [107]:
#生成日期序列
date_range=pd.date_range(start=start_date,end=end_date,freq='D')
df_dates=pd.DataFrame({'date':date_range})

In [108]:
#格式化日期
df_dates['Date']=df_dates['date'].dt.strftime("%Y%m%d")
df_dates['Date']=pd.to_datetime(df_dates['Date'])

In [109]:
df_dates=df_dates['Date']

In [110]:
df_dates

0      2005-12-31
1      2006-01-01
2      2006-01-02
3      2006-01-03
4      2006-01-04
          ...    
7113   2025-06-22
7114   2025-06-23
7115   2025-06-24
7116   2025-06-25
7117   2025-06-26
Name: Date, Length: 7118, dtype: datetime64[ns]

In [15]:
df2=df1[df1['S_INFO_WINDCODE']=='600000.SH']
df2['ACTUAL_ANN_DT']=pd.to_datetime(df2['ACTUAL_ANN_DT'])

In [16]:
#合并两个数据框
df_merged=pd.merge_asof(df_dates,df2,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')

In [17]:
df_merged

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
0,2005-12-31,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
1,2006-01-01,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
2,2006-01-02,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
3,2006-01-03,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
4,2006-01-04,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6936,2024-12-27,{1739EDB8-2E8A-8BCD-E063-2001C80A4558},600000.SH,1600000,20240430,2024-04-30,20231231,408001000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表,2024-04-29 18:01:55,0
6937,2024-12-28,{1739EDB8-2E8A-8BCD-E063-2001C80A4558},600000.SH,1600000,20240430,2024-04-30,20231231,408001000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表,2024-04-29 18:01:55,0
6938,2024-12-29,{1739EDB8-2E8A-8BCD-E063-2001C80A4558},600000.SH,1600000,20240430,2024-04-30,20231231,408001000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表,2024-04-29 18:01:55,0
6939,2024-12-30,{1739EDB8-2E8A-8BCD-E063-2001C80A4558},600000.SH,1600000,20240430,2024-04-30,20231231,408001000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表,2024-04-29 18:01:55,0


In [18]:
df2

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
458,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,2007-03-24,20061231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0
1658,{FAB2BFD7-6A7F-4C45-A091-D72A0E8BA983},600000.SH,1600000,20080228,2008-02-28,20071231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
3995,{F582D3F0-9CE6-45BE-A64F-D3662DAD0324},600000.SH,1600000,20090410,2009-04-10,20081231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
5766,{FB15E432-6AC5-432D-BC2C-CBF275F77652},600000.SH,1600000,20100407,2010-04-07,20091231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
7585,{6A51EC4A-2A97-49C7-B2F5-F5FEAD11F304},600000.SH,1600000,20110330,2011-03-30,20101231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:34:37,0
9135,{14E33115-1AAC-467F-8BAB-42A74AAB7E93},600000.SH,1600000,20120316,2012-03-16,20111231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:34:42,0
11360,{E059C8B0-2751-45AC-8FD6-B884CEB0B315},600000.SH,1600000,20130314,2013-03-14,20121231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:38:20,0
14137,{312D0709-C736-4BA0-9ABA-1E01AF1C1E2B},600000.SH,1600000,20140320,2014-03-20,20131231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:18:40,0
16655,{CCA555A3-A939-4977-A29C-A800CCD5AFE2},600000.SH,1600000,20150319,2015-03-19,20141231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:01:40,0


### 对齐年报数据到日级别

In [19]:
#获取所有股票代码
stock_ids=df1['S_INFO_WINDCODE'].unique()

In [20]:
results=[]

In [21]:
for stock_id in tqdm(stock_ids,desc='Processing stocks'):
    #提取当前股票的年报数据并按实际公布日期排序
    stock_annual=df1[df1['S_INFO_WINDCODE']==stock_id].sort_values('ACTUAL_ANN_DT')
    
    stock_annual['ACTUAL_ANN_DT']=pd.to_datetime(stock_annual['ACTUAL_ANN_DT'])
    
    #合并两个数据框
    df_merged=pd.merge_asof(df_dates,stock_annual,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')
    
    results.append(df_merged)
    

Processing stocks: 100%|██████████| 5680/5680 [01:02<00:00, 90.92it/s] 


In [22]:
#合并结果
final_result=pd.concat(results,ignore_index=True)

In [23]:
final_result.shape

(39424880, 123)

In [24]:
final_result.sort_values(['Date','S_INFO_WINDCODE'],inplace=True)

In [25]:
final_result=final_result.dropna(subset=['S_INFO_WINDCODE'])

In [26]:
final_result.head(10)

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
61,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
7002,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
13943,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
20884,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
27825,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
34766,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
41707,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
48648,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
55589,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
62530,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0


In [27]:
final_result.shape

(39078400, 123)

### 年报数据，对齐过去一年，优先调整

In [195]:
# 获取现金流量表中的年报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOWHIS where  REPORT_PERIOD like '%1231' and STATEMENT_TYPE in (408001000,408004000,408050000,408029000,408031000,408037000,408046000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df3 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [196]:
df3.shape

(122620, 122)

In [197]:
df4=df3[df3['S_INFO_WINDCODE']=='600000.SH']
df4
# df2['ACTUAL_ANN_DT']=pd.to_datetime(df2['ACTUAL_ANN_DT'])

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
9,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,20070324,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
457,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,20070324,20061231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0
1655,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,20080228,20061231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
3243,{FAB2BFD7-6A7F-4C45-A091-D72A0E8BA983},600000.SH,1600000,20080228,20080228,20071231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
5593,{C39AF249-F4C7-4DFC-80E0-01E18EAB975D},600000.SH,1600000,20090410,20090410,20071231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
7219,{F582D3F0-9CE6-45BE-A64F-D3662DAD0324},600000.SH,1600000,20090410,20090410,20081231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
8939,{AA8E7DF4-1C8A-48C3-BCE4-4D65E9DAD2EE},600000.SH,1600000,20100407,20100407,20081231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
10716,{FB15E432-6AC5-432D-BC2C-CBF275F77652},600000.SH,1600000,20100407,20100407,20091231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
12519,{BBD4F7FF-CF19-4395-AF14-F414F1E358AC},600000.SH,1600000,20110330,20110330,20091231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0


In [90]:
df4=df4[df4['STATEMENT_TYPE']=='408001000']

In [91]:
df4

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
457,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,20070324,20061231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0
3243,{FAB2BFD7-6A7F-4C45-A091-D72A0E8BA983},600000.SH,1600000,20080228,20080228,20071231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
7219,{F582D3F0-9CE6-45BE-A64F-D3662DAD0324},600000.SH,1600000,20090410,20090410,20081231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
10716,{FB15E432-6AC5-432D-BC2C-CBF275F77652},600000.SH,1600000,20100407,20100407,20091231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
14703,{6A51EC4A-2A97-49C7-B2F5-F5FEAD11F304},600000.SH,1600000,20110330,20110330,20101231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:34:37,0
18601,{14E33115-1AAC-467F-8BAB-42A74AAB7E93},600000.SH,1600000,20120316,20120316,20111231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:34:42,0
23312,{E059C8B0-2751-45AC-8FD6-B884CEB0B315},600000.SH,1600000,20130314,20130314,20121231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:38:20,0
28645,{312D0709-C736-4BA0-9ABA-1E01AF1C1E2B},600000.SH,1600000,20140320,20140320,20131231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:18:40,0
33822,{CCA555A3-A939-4977-A29C-A800CCD5AFE2},600000.SH,1600000,20150319,20150319,20141231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-15 18:01:40,0


In [92]:
df4['ACTUAL_ANN_DT']=pd.to_datetime(df4['ACTUAL_ANN_DT'])

In [93]:
#合并两个数据框
df_merged=pd.merge_asof(df_dates,df4,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')

In [94]:
df_merged=df_merged.dropna(subset=['S_INFO_WINDCODE'])
df_merged

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
61,2006-03-02,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
62,2006-03-03,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
63,2006-03-04,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
64,2006-03-05,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
65,2006-03-06,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7114,2025-06-23,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7115,2025-06-24,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7116,2025-06-25,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0


In [153]:
df_yoy=df4.copy()
df_yoy['ACTUAL_ANN_DT']=pd.to_datetime(df_yoy['ACTUAL_ANN_DT'])
# df_yoy['PREV_YEAR_REPORT_PERIOD'] = (df_yoy['REPORT_PERIOD'].str[:4].astype(int) - 1).astype(str) + '1231'

In [154]:
# 先按STATEMENT_TYPE排序，确保408004000(调整)排在前面
df_yoy = df_yoy.sort_values(
    ['REPORT_PERIOD', 'STATEMENT_TYPE'], 
    ascending=[True, False]  # 408004000会排在408001000前面
)

# 去除重复报告期，保留优先级高的
df_yoy = df_yoy.drop_duplicates('REPORT_PERIOD', keep='first')

In [162]:
df_yoy=df_yoy[:-1]
df_yoy

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
9,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1655,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,2008-02-28,20061231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
5593,{C39AF249-F4C7-4DFC-80E0-01E18EAB975D},600000.SH,1600000,20090410,2009-04-10,20071231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
8939,{AA8E7DF4-1C8A-48C3-BCE4-4D65E9DAD2EE},600000.SH,1600000,20100407,2010-04-07,20081231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
12519,{BBD4F7FF-CF19-4395-AF14-F414F1E358AC},600000.SH,1600000,20110330,2011-03-30,20091231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
16245,{21FA1A4D-54CA-41A4-B1DC-FB4A245C4026},600000.SH,1600000,20120316,2012-03-16,20101231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
20800,{82367AE3-1F82-4769-A1F2-A9AAAF609BAC},600000.SH,1600000,20130314,2013-03-14,20111231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
26088,{991E9FD5-63E3-4E8C-AF1A-FD9FF24F2014},600000.SH,1600000,20140320,2014-03-20,20121231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:38:20,0
31115,{32A151D8-771C-42E9-8798-F5D8CBC59240},600000.SH,1600000,20150319,2015-03-19,20131231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:18:40,0
37359,{2F6B7BD9-48A9-0E27-E053-1001C80A108F},600000.SH,1600000,20160407,2016-04-07,20141231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:01:40,0


In [152]:
df4

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
2,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,20060302,20051231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
9,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,20070324,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
457,{6EA8B473-816C-40EB-96B2-28C7B579D3D4},600000.SH,1600000,20070324,20070324,20061231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2024-10-08 22:31:41,0
1655,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,20080228,20061231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
3243,{FAB2BFD7-6A7F-4C45-A091-D72A0E8BA983},600000.SH,1600000,20080228,20080228,20071231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
5593,{C39AF249-F4C7-4DFC-80E0-01E18EAB975D},600000.SH,1600000,20090410,20090410,20071231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
7219,{F582D3F0-9CE6-45BE-A64F-D3662DAD0324},600000.SH,1600000,20090410,20090410,20081231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
8939,{AA8E7DF4-1C8A-48C3-BCE4-4D65E9DAD2EE},600000.SH,1600000,20100407,20100407,20081231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
10716,{FB15E432-6AC5-432D-BC2C-CBF275F77652},600000.SH,1600000,20100407,20100407,20091231,408001000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表,2019-03-18 18:49:25,0
12519,{BBD4F7FF-CF19-4395-AF14-F414F1E358AC},600000.SH,1600000,20110330,20110330,20091231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0


In [163]:
df_yoy['ACTUAL_ANN_DT']=pd.to_datetime(df_yoy['ACTUAL_ANN_DT'])

In [164]:
# 3. 用 merge_asof 对齐时间轴（与 df_merged 相同逻辑）
df_prev_year = pd.merge_asof(df_dates,df_yoy,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')

In [167]:
df_prev_year=df_prev_year.dropna(subset=['S_INFO_WINDCODE'])

In [168]:
df_prev_year

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
448,2007-03-24,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
449,2007-03-25,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
450,2007-03-26,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
451,2007-03-27,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
452,2007-03-28,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7114,2025-06-23,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7115,2025-06-24,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7116,2025-06-25,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0


In [171]:
df_merged[300:]

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
361,2006-12-27,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
362,2006-12-28,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
363,2006-12-29,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
364,2006-12-30,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
365,2006-12-31,{BFAEE18B-E6AD-4BEC-96E0-2691DDB9CACD},600000.SH,1600000,20070324,2006-03-02,20051231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表,2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7114,2025-06-23,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7115,2025-06-24,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0
7116,2025-06-25,{3166C4B7-0169-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20241231,408001000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,6.943700e+10,NaN,合并报表,2025-03-28 23:01:43,0


#### 构造前两年的时间轴数据

In [189]:
df_yoy

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
9,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1655,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,2008-02-28,20061231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
5593,{C39AF249-F4C7-4DFC-80E0-01E18EAB975D},600000.SH,1600000,20090410,2009-04-10,20071231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
8939,{AA8E7DF4-1C8A-48C3-BCE4-4D65E9DAD2EE},600000.SH,1600000,20100407,2010-04-07,20081231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
12519,{BBD4F7FF-CF19-4395-AF14-F414F1E358AC},600000.SH,1600000,20110330,2011-03-30,20091231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
16245,{21FA1A4D-54CA-41A4-B1DC-FB4A245C4026},600000.SH,1600000,20120316,2012-03-16,20101231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
20800,{82367AE3-1F82-4769-A1F2-A9AAAF609BAC},600000.SH,1600000,20130314,2013-03-14,20111231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
26088,{991E9FD5-63E3-4E8C-AF1A-FD9FF24F2014},600000.SH,1600000,20140320,2014-03-20,20121231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:38:20,0
31115,{32A151D8-771C-42E9-8798-F5D8CBC59240},600000.SH,1600000,20150319,2015-03-19,20131231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:18:40,0
37359,{2F6B7BD9-48A9-0E27-E053-1001C80A108F},600000.SH,1600000,20160407,2016-04-07,20141231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:01:40,0


In [190]:
#创建同比DataFrame
df_yoy1 = df_yoy.copy()

# 将日期向前推一年
df_yoy1['ACTUAL_ANN_DT'] = df_yoy1['ACTUAL_ANN_DT'] + pd.DateOffset(years=2)

df_yoy1

,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
9,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2009-03-24,20051231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1655,{E85DFC95-2AEE-4E6B-9AA9-EEE711A06649},600000.SH,1600000,20080228,2010-02-28,20061231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
5593,{C39AF249-F4C7-4DFC-80E0-01E18EAB975D},600000.SH,1600000,20090410,2011-04-10,20071231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
8939,{AA8E7DF4-1C8A-48C3-BCE4-4D65E9DAD2EE},600000.SH,1600000,20100407,2012-04-07,20081231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
12519,{BBD4F7FF-CF19-4395-AF14-F414F1E358AC},600000.SH,1600000,20110330,2013-03-30,20091231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:49:25,0
16245,{21FA1A4D-54CA-41A4-B1DC-FB4A245C4026},600000.SH,1600000,20120316,2014-03-16,20101231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
20800,{82367AE3-1F82-4769-A1F2-A9AAAF609BAC},600000.SH,1600000,20130314,2015-03-14,20111231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-18 18:34:42,0
26088,{991E9FD5-63E3-4E8C-AF1A-FD9FF24F2014},600000.SH,1600000,20140320,2016-03-20,20121231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:38:20,0
31115,{32A151D8-771C-42E9-8798-F5D8CBC59240},600000.SH,1600000,20150319,2017-03-19,20131231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:18:40,0
37359,{2F6B7BD9-48A9-0E27-E053-1001C80A108F},600000.SH,1600000,20160407,2018-04-07,20141231,408004000,CNY,NaN,NaN,...,NaN,NaN,0,NaN,None,NaN,NaN,合并报表(调整),2019-03-15 18:01:40,0


In [192]:
df_prev_year1 = pd.merge_asof(df_dates,df_yoy1,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')
# 恢复原始日期
df_prev_year1['ACTUAL_ANN_DT'] = df_prev_year1['ACTUAL_ANN_DT'] - pd.DateOffset(years=2)

In [193]:
df_prev_year1=df_prev_year1.dropna(subset=['S_INFO_WINDCODE'])

In [194]:
df_prev_year1

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
1179,2009-03-24,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1180,2009-03-25,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1181,2009-03-26,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1182,2009-03-27,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
1183,2009-03-28,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{F9998F17-352F-4928-E053-2001C80A0823},600000.SH,1600000,20230419,2023-04-19,20211231,408004000,CNY,NaN,...,NaN,4.870900e+10,0.0,NaN,None,7.833100e+10,NaN,合并报表(调整),2023-04-18 18:50:48,0
7114,2025-06-23,{F9998F17-352F-4928-E053-2001C80A0823},600000.SH,1600000,20230419,2023-04-19,20211231,408004000,CNY,NaN,...,NaN,4.870900e+10,0.0,NaN,None,7.833100e+10,NaN,合并报表(调整),2023-04-18 18:50:48,0
7115,2025-06-24,{F9998F17-352F-4928-E053-2001C80A0823},600000.SH,1600000,20230419,2023-04-19,20211231,408004000,CNY,NaN,...,NaN,4.870900e+10,0.0,NaN,None,7.833100e+10,NaN,合并报表(调整),2023-04-18 18:50:48,0
7116,2025-06-25,{F9998F17-352F-4928-E053-2001C80A0823},600000.SH,1600000,20230419,2023-04-19,20211231,408004000,CNY,NaN,...,NaN,4.870900e+10,0.0,NaN,None,7.833100e+10,NaN,合并报表(调整),2023-04-18 18:50:48,0


In [188]:
df_prev_year

,Date,OBJECT_ID,S_INFO_WINDCODE,S_INFO_COMPCODE,ANN_DT,ACTUAL_ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,...,TOT_BAL_NETCASH_INC_UNDIR,S_DISMANTLE_CAPITAL_ADD_NET,IS_CALCULATION,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,STATEMENT_TYPE_WIND,OPDATE,OPMODE
448,2007-03-24,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
449,2007-03-25,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
450,2007-03-26,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
451,2007-03-27,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
452,2007-03-28,{62578663-E788-4FD2-B660-D1EAD68672D3},600000.SH,1600000,20070324,2007-03-24,20051231,408004000,CNY,NaN,...,NaN,NaN,0.0,NaN,None,NaN,NaN,合并报表(调整),2019-06-18 23:42:47,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,2025-06-22,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7114,2025-06-23,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7115,2025-06-24,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
7116,2025-06-25,{3166C4B7-016D-2011-E063-1F01C80ADDDE},600000.SH,1600000,20250329,2025-03-29,20231231,408004000,CNY,NaN,...,NaN,1.043630e+11,0.0,NaN,None,7.675400e+10,NaN,合并报表(调整),2025-03-28 23:01:43,0
